In [3]:
import pandas as pd

df = pd.read_csv("../13k-recipes.csv")
recipes = df['Title'].astype(str).tolist()


In [4]:
from transformers import DistilBertTokenizer, DistilBertModel
import torch

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertModel.from_pretrained('distilbert-base-uncased')
model.eval()


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSdpaAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [5]:
import numpy as np

def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings[0].numpy()


In [6]:
from tqdm import tqdm

recipe_embeddings = []

for recipe in tqdm(recipes):
    embedding = get_embedding(recipe)
    recipe_embeddings.append(embedding)

recipe_embeddings = np.array(recipe_embeddings)


100%|██████████| 13501/13501 [11:15<00:00, 19.98it/s]


In [7]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')
recipe_embeddings = model.encode(recipes, show_progress_bar=True)
def recommend_recipes(query, top_k=3):
    query_vec = model.encode([query])
    similarities = cosine_similarity(query_vec, recipe_embeddings)[0]
    top_indices = similarities.argsort()[-top_k:][::-1]
    return [(recipes[i], similarities[i]) for i in top_indices]


Batches:   0%|          | 0/422 [00:00<?, ?it/s]C:\Users\tapuc\Desktop\handmade_backup\licenta\TransformereTest\.venv\Lib\site-packages\torch\nn\modules\module.py:1784: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Batches: 100%|██████████| 422/422 [00:29<00:00, 14.18it/s]


In [10]:
user_input = "I am craving a pistachio sweet"
recommendations = recommend_recipes(user_input)

print("Top 3 recommended desserts:\n")
for title, score in recommendations:
    print(f"{title} (score: {score:.2f})")


Top 3 recommended desserts:

Spiced Pistachios (score: 0.70)
Pistachio Cake (score: 0.70)
Pistachio Cake (score: 0.70)


C:\Users\tapuc\Desktop\handmade_backup\licenta\TransformereTest\.venv\Lib\site-packages\torch\nn\modules\module.py:1784: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [8]:
import numpy as np

def evaluate_model(test_cases, top_k=3):
    precisions = []

    print(f"{'Query':<30} | {'Top Result':<40} | {'Match?'}")
    print("-" * 80)

    for query, expected_keywords in test_cases.items():
        results = recommend_recipes(query, top_k=top_k)
        relevant_count = 0
        found_match = False
        keywords = [k.lower() for k in expected_keywords]

        for title, score in results:
            title_lower = title.lower()
            if any(keyword in title_lower for keyword in keywords):
                relevant_count += 1

        precision = relevant_count / top_k
        precisions.append(precision)

        top_result_title = results[0][0] if results else "No results"
        is_match = "good" if any(k in top_result_title.lower() for k in keywords) else "bad"
        print(f"{query:<30} | {top_result_title[:37]:<40}... | {is_match}")

    mean_precision = np.mean(precisions)
    return mean_precision

test_set = {
    "pistachio dessert": ["pistachio"],
    "chocolate cake": ["chocolate", "cocoa", "fudge"],
    "grilled chicken": ["chicken", "poultry"],
    "spicy tacos": ["taco", "mexican", "tortilla"],
    "italian pasta": ["pasta", "spaghetti", "fettuccine", "lasagna"],
    "healthy salad": ["salad", "vegetable", "kale", "spinach"]
}

avg_precision = evaluate_model(test_set, top_k=3)

print("\n" + "="*40)
print(f"Mean Precision@3: {avg_precision * 100:.2f}%")
print("="*40)

Query                          | Top Result                               | Match?
--------------------------------------------------------------------------------
pistachio dessert              | Pistachio Cake                          ... | good


C:\Users\tapuc\Desktop\handmade_backup\licenta\TransformereTest\.venv\Lib\site-packages\torch\nn\modules\module.py:1784: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
C:\Users\tapuc\Desktop\handmade_backup\licenta\TransformereTest\.venv\Lib\site-packages\torch\nn\modules\module.py:1784: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


chocolate cake                 | Chocolate Cake                          ... | good
grilled chicken                | Grilled Lemon Chicken                   ... | good
spicy tacos                    | Spicy Black Bean and Corn Tacos         ... | good


C:\Users\tapuc\Desktop\handmade_backup\licenta\TransformereTest\.venv\Lib\site-packages\torch\nn\modules\module.py:1784: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
C:\Users\tapuc\Desktop\handmade_backup\licenta\TransformereTest\.venv\Lib\site-packages\torch\nn\modules\module.py:1784: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
C:\Users\tapuc\Desktop\handmade_backup\licenta\TransformereTest\.venv\Lib\site-packages\torch\nn\modules\module.py:1784: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


italian pasta                  | Fresh Pasta                             ... | good
healthy salad                  | Salad for Breakfast                     ... | good

Mean Precision@3: 100.00%


C:\Users\tapuc\Desktop\handmade_backup\licenta\TransformereTest\.venv\Lib\site-packages\torch\nn\modules\module.py:1784: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [9]:
import pickle

data_to_save = {
    "embeddings": recipe_embeddings,  # Matricea numpy
    "titles": recipes,               # Lista de titluri (strings)
    "model_name": "all-MiniLM-L6-v2" # Ca să știm ce model să încărcăm
}

filename = "../recipe_recommendation_data.pkl"
with open(filename, "wb") as f:
    pickle.dump(data_to_save, f)

print(f"Fisierul '{filename}' a fost generat cu succes!")

Fisierul 'recipe_recommendation_data.pkl' a fost generat cu succes!


Pentru a asigura o acuratețe ridicată și o viteză optimă a aplicației, am ales o abordare bazată pe Transfer Learning și Semantic Search (Căutare Semantică).

În loc să antrenez o rețea neurală simplă de la zero, care nu ar fi înțeles contextul lingvistic, am utilizat un model de tip Transformer (DistilBERT) pre-antrenat pe un corpus masiv de date. Am folosit acest model pentru a transforma textul rețetelor și input-ul utilizatorului în vectori numerici (embeddings) într-un spațiu multidimensional.

Modelul utilizat: "Am ales DistilBERT (o versiune distilată a BERT) deoarece oferă un echilibru ideal între performanță (înțelegerea limbajului natural) și eficiență computațională (viteză de răspuns în aplicația web)."

Tehnica de reprezentare: "Am implementat o tehnică de Vector Space Model. Fiecare rețetă este convertită într-un vector dens (embedding). Astfel, sistemul nu caută doar cuvinte cheie (ca un simplu CTRL+F), ci înțelege sensul cuvintelor (de exemplu, știe că 'desert' și 'dulce' sunt similare semantic)."

Algoritmul de potrivire: "Pentru recomandare, am folosit Cosine Similarity (Similaritatea Cosinusului) pentru a măsura distanța unghiulară dintre vectorul cererii utilizatorului și vectorii rețetelor."

DA, 100% este AI. Și nu doar „AI”, ci este State-of-the-Art NLP (Natural Language Processing).